In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from pycontrails import Fleet

In [2]:
# EGWP 100
dff = pd.read_parquet("../data/filed_trajectories_EAGWP100.parquet")
fleetf = Fleet(data=dff)

dfo = pd.read_parquet("../data/optimised_trajectories_EAGWP100.parquet")
fleeto = Fleet(data=dfo)

In [7]:
metaf = pd.read_parquet("../data/filed_meta.parquet")
metao = pd.read_parquet("../data/optimised_meta.parquet")

In [9]:
cols = ["flight_id", "flight_time", "flown_distance", "fuel_use", "co2eq_gwp_pulse_100_co2", "co2eq_gwp_pulse_100_contrails", "co2eq_gwp_pulse_100_total"]
meta_diff = pd.merge(
    metaf[cols],
    metao[cols],
    on="flight_id",
    suffixes=("_f", "_o")
)

In [ ]:
# Thales only (forecast)
# TODO


In [ ]:
# Thales only (which is already based on reanalysis)
# contrails
print(">>> Contrail")
print(meta_diff.query("co2eq_gwp_pulse_100_contrails_o > co2eq_gwp_pulse_100_contrails_f").shape[0], "failed", int(746/4112*100), "%")
print(meta_diff.query("co2eq_gwp_pulse_100_contrails_o < co2eq_gwp_pulse_100_contrails_f").shape[0], "success", int(3352/4112*100), "%")
print(meta_diff.query("co2eq_gwp_pulse_100_contrails_o == co2eq_gwp_pulse_100_contrails_f").shape[0], "same", int(14/4112*100), "%")

# contrails + CO2
print("\n>>> Contrail + CO2")
print(meta_diff.query("co2eq_gwp_pulse_100_total_o > co2eq_gwp_pulse_100_total_f").shape[0], "failed", int(866/4112*100), "%")
print(meta_diff.query("co2eq_gwp_pulse_100_total_o < co2eq_gwp_pulse_100_total_f").shape[0], "success", int(3246/4112*100), "%")
print(meta_diff.query("co2eq_gwp_pulse_100_total_o == co2eq_gwp_pulse_100_total_f").shape[0], "same", int(0/4112*100), "%")

print("\n>>> Changes")
print(3352-3246, "less successes")
print(866-746, "more fails")

>>> Contrail
746 failed 18 %
3352 success 81 %
14 same 0 %

>>> Contrail + CO2
866 failed 21 %
3246 success 78 %
0 same 0 %

>>> Changes
106 less successes
120 more fails


In [44]:
from cane.utils import mask_by_marker

mask_by_marker(fleetf, ["NOx", "O3", "CH4", "H2O"])
mask_by_marker(fleeto, ["NOx", "O3", "CH4", "H2O"])

from cane.utils import mask_by_validity_range

bounds = [150, 350]
mask_by_validity_range(fleetf, ["NOx", "O3", "CH4", "H2O"], bounds)
mask_by_validity_range(fleeto, ["NOx", "O3", "CH4", "H2O"], bounds)

dff = fleetf.dataframe
dfo = fleeto.dataframe

dff["CO2_CoCiP"] = dff["CO2"] + dff["CoCiP"]
dfo["CO2_CoCiP"] = dfo["CO2"] + dfo["CoCiP"]

dff["NOx_H2O"] = dff["NOx"] + dff["H2O"]
dfo["NOx_H2O"] = dfo["NOx"] + dfo["H2O"]

dff["Total"] = dff["CO2"] + dff["CoCiP"] + dff["NOx"] + dff["H2O"]
dfo["Total"] = dfo["CO2"] + dfo["CoCiP"] + dfo["NOx"] + dfo["H2O"]

Marked rows filtered out for NOx
Marked rows filtered out for O3
Marked rows filtered out for CH4
Marked rows filtered out for H2O
Marked rows filtered out for NOx
Marked rows filtered out for O3
Marked rows filtered out for CH4
Marked rows filtered out for H2O
Bounds of [150, 350] in place for NOx
Bounds of [150, 350] in place for O3
Bounds of [150, 350] in place for CH4
Bounds of [150, 350] in place for H2O
Bounds of [150, 350] in place for NOx
Bounds of [150, 350] in place for O3
Bounds of [150, 350] in place for CH4
Bounds of [150, 350] in place for H2O


In [45]:
from cane.utils import df_diff

# create aggregates (total)
org_sum = (
    dff[["flight_id", "fuel_burn", "nox", "co2", "CO2", "NOx", "CoCiP", "H2O", "CO2_CoCiP", "NOx_H2O", "Total"]]
    .groupby(["flight_id"]).sum().reset_index()
)
opt_sum = (
    dfo[["flight_id", "fuel_burn", "nox", "co2", "CO2", "NOx", "CoCiP", "H2O", "CO2_CoCiP", "NOx_H2O", "Total"]]
    .groupby(["flight_id"]).sum().reset_index()
)
my_diff = df_diff(org_sum, opt_sum, on=["flight_id"], keep_originals=True)
my_diff = pd.merge(
    my_diff,
    dff.groupby("flight_id").day.first(),
    on="flight_id",
)

In [49]:
# Evaluation only
# contrails
print(">>> Contrail")
print(my_diff.query("CoCiP_optimised > CoCiP_filed").shape[0], "failed", int(445/4112*100), "%")
print(my_diff.query("CoCiP_optimised < CoCiP_filed").shape[0], "success", int(3626/4112*100), "%")
print(my_diff.query("CoCiP_optimised == CoCiP_filed").shape[0], "same", int(41/4112*100), "%")

# contrails + CO2
print("\n>>> Contrail + CO2")
print(my_diff.query("CO2_CoCiP_optimised > CO2_CoCiP_filed").shape[0], "failed", int(609/4112*100), "%")
print(my_diff.query("CO2_CoCiP_optimised < CO2_CoCiP_filed").shape[0], "success", int(3503/4112*100), "%")
print(my_diff.query("CO2_CoCiP_optimised == CO2_CoCiP_filed").shape[0], "same", int(0/4112*100), "%")

print("\n>>> Changes")
print(3626-3503, "less successes")
print(609-445, "more fails")

>>> Contrail
445 failed 10 %
3626 success 88 %
41 same 0 %

>>> Contrail + CO2
609 failed 14 %
3503 success 85 %
0 same 0 %

>>> Changes
123 less successes
164 more fails


In [58]:
# CO2
print("\n>>> CO2")
print(my_diff.query("CO2_optimised > CO2_filed").shape[0], "failed", int(3920/4112*100), "%")
print(my_diff.query("CO2_optimised < CO2_filed").shape[0], "success", int(192/4112*100), "%")
print(my_diff.query("CO2_optimised == CO2_filed").shape[0], "same", int(0/4112*100), "%")

# NOx
print("\n>>> NOx")
print(my_diff.query("NOx_optimised > NOx_filed").shape[0], "failed", int(2687/4112*100), "%")
print(my_diff.query("NOx_optimised < NOx_filed").shape[0], "success", int(1324/4112*100), "%")
print(my_diff.query("NOx_optimised == NOx_filed").shape[0], "same", int(101/4112*100), "%")

# H2O
print("\n>>> H2O")
print(my_diff.query("H2O_optimised > H2O_filed").shape[0], "failed", int(1572/4112*100), "%")
print(my_diff.query("H2O_optimised < H2O_filed").shape[0], "success", int(2439/4112*100), "%")
print(my_diff.query("H2O_optimised == H2O_filed").shape[0], "same", int(101/4112*100), "%")

# NOx + H2O
print("\n>>> NOx + H2O")
print(my_diff.query("NOx_H2O_optimised > NOx_H2O_filed").shape[0], "failed", int(2570/4112*100), "%")
print(my_diff.query("NOx_H2O_optimised < NOx_H2O_filed").shape[0], "success", int(1441/4112*100), "%")
print(my_diff.query("NOx_H2O_optimised == NOx_H2O_filed").shape[0], "same", int(101/4112*100), "%")

# Total
print("\n>>> Contrail + CO2 + NOx + H2O")
print(my_diff.query("Total_optimised > Total_filed").shape[0], "failed", int(627/4112*100), "%")
print(my_diff.query("Total_optimised < Total_filed").shape[0], "success", int(3485/4112*100), "%")
print(my_diff.query("Total_optimised == Total_filed").shape[0], "same", int(0/4112*100), "%")


>>> CO2
3920 failed 95 %
192 success 4 %
0 same 0 %

>>> NOx
2687 failed 65 %
1324 success 32 %
101 same 2 %

>>> H2O
1572 failed 38 %
2439 success 59 %
101 same 2 %

>>> NOx + H2O
2570 failed 62 %
1441 success 35 %
101 same 2 %

>>> Contrail + CO2 + NOx + H2O
627 failed 15 %
3485 success 84 %
0 same 0 %
